In [1]:
# ============================================================
# 0.  Imports  (plain Colab already has SymPy 1.8-1.12 & NumPy 1.26)
# ============================================================
import sympy as sp
import numpy as np
from itertools import combinations


# ------------------------------------------------------------
# 0-b. Optional PSLQ loader (three fallbacks → else None)
# ------------------------------------------------------------
def load_pslq():
    """
    Tries to load the 'pslq' function from one of three places:
      1) 'sympy.numerics.number_complexity' (SymPy 1.13+ WITH numerics),
      2) 'sympy.ntheory' (SymPy ≤1.12 legacy),
      3) 'sympy_numerics.number_complexity' (external ‘sympy-numerics’ wheel).
    Returns PSLQ function or None.
    """
    PSLQ = None
    for _path in [
        "sympy.numerics.number_complexity",  # SymPy 1.13+ WITH numerics
        "sympy.ntheory",                     # SymPy ≤1.12 legacy
        "sympy_numerics.number_complexity",  # external ‘sympy-numerics’ wheel
    ]:
        try:
            PSLQ = __import__(_path, fromlist=["pslq"]).pslq
            break
        except Exception:
            pass
    return PSLQ

PSLQ = load_pslq()


# ============================================================
# 1.  Symbols, pretty names, CODATA (wrapped in sp.Float)
# ============================================================
ℏ, c, G, k_B, α = sp.symbols('hbar c G k_B alpha', positive=True)
ε0, μ0, e_q, σ_SB, h = sp.symbols('epsilon_0 mu_0 e sigma_SB h', positive=True)

KNOWN = {
    sp.pi : 'π',
    2*sp.pi : '2π',
    sp.E : 'e',
    ℏ : 'ℏ',
    h  : 'h',
    c  : 'c',
    G  : 'G',
    k_B: 'k_B',
    α  : 'α',
    ε0 : 'ε0',
    μ0 : 'μ0',
    e_q: 'e (charge)',
    σ_SB: 'σ_SB',
}

CODATA = {
    ℏ  : 1.054_571_817e-34,
    h   : 6.626_070_15e-34,
    c   : 2.997_924_58e8,
    G   : 6.674_30e-11,
    k_B : 1.380_649e-23,
    ε0  : 8.854_187_8128e-12,
    μ0  : 1.256_637_06212e-6,
    e_q : 1.602_176_634e-19,
    σ_SB: 5.670_374_419e-8,
    α   : 7.297_352_5693e-3,
}
CODATA = {k: sp.Float(v) for k, v in CODATA.items()}


# ============================================================
# 2.  Equation bank (edit/extend as you like)
# ============================================================
E, m, F, a, q1, q2, r, ν, T = sp.symbols('E m F a q1 q2 r nu T')  # misc symbols
λ = sp.symbols('lambda')
ψ_t, ψ_xx = sp.symbols('psi_t psi_xx')
Z0 = sp.symbols('Z_0')

EQUATIONS = {
    "Einstein mass–energy" : sp.Eq(E, m*c**2),
    "Newton II"            : sp.Eq(F, m*a),
    "Coulomb law"          : sp.Eq(F, 1/(4*sp.pi*ε0)*q1*q2/r**2),
    "de Broglie λ"         : sp.Eq(λ, ℏ/(m*c)),
    "Schrödinger (free)"   : sp.Eq(sp.I*ℏ*ψ_t, -ℏ**2/(2*m)*ψ_xx),
    "Planck E=hν"          : sp.Eq(E, h*ν),
    "Boltzmann kT"         : sp.Eq(E, k_B*T),
    "Stefan–Boltzmann"     : sp.Eq(σ_SB, sp.pi**2*k_B**4/(60*ℏ**3*c**2)),
    "Einstein A↔B"         : sp.Eq(sp.Symbol('B_12'), sp.Symbol('A_21')*c**3/(8*sp.pi*h*ν**3)),
    "Fine-structure α"     : sp.Eq(α, e_q**2/(4*sp.pi*ε0*ℏ*c)),
    "Vacuum impedance"     : sp.Eq(Z0, μ0*c),
    "Rydberg constant"     : sp.Eq(sp.Symbol('R_inf'), α**2*m*c/(2*h)),
}


# ============================================================
# 3.  Helper functions
# ============================================================
def constants(expr):
    """
    Return a set of recognized constants (symbolic or numeric) that appear in 'expr'.
    This includes:
      • Known symbols (ℏ, c, etc.)
      • Numeric coefficients in the expression
    """
    out = set()
    # Step 1: Expand the expression into a sum of terms, then factor each term
    # to identify numeric factors
    for term in sp.Add.make_args(expr.expand()):
        for fac in sp.factor_terms(term, radical=True).as_ordered_factors():
            if fac.is_Number and fac != 1:
                out.add(sp.nsimplify(fac))
    # Step 2: If a known symbol is in the expression, add it
    for s in expr.free_symbols:
        if s in KNOWN:
            out.add(s)
    # Step 3: Another pass checking .has(k)
    for k in KNOWN:
        if expr.has(k):
            out.add(k)
    return out


def pretty(symbol_set):
    """
    Return a readable string for a set of constants, matching the 'KNOWN' dict.
    """
    return ", ".join(KNOWN.get(x, str(x))
                     for x in sorted(symbol_set, key=lambda z: (not z.is_Number, str(z))))


def num(expr):
    """
    Numerically evaluate a Sympy expression by substituting CODATA values
    for the known constants.
    """
    return expr.xreplace(CODATA).evalf()


def coeffs(expr):
    """
    Extract a list of the numeric coefficients (floats) from an expression.
    """
    out = []
    # Expand the expression into a sum of terms
    for term in sp.Add.make_args(expr.expand()):
        # Factor each term
        for fac in sp.factor_terms(term, radical=True).as_ordered_factors():
            if fac.is_Number and fac != 1:
                out.append(float(fac))
    return out


# ============================================================
# 4.  Inventory & overlap
# ============================================================
def constant_inventory(equations):
    """
    Prints out which constants appear in each equation.
    Returns a dict: {Equation name -> set_of_constants}.
    """
    tbl = {}
    for eq_name, eq in equations.items():
        lhs_consts = constants(eq.lhs)
        rhs_consts = constants(eq.rhs)
        tbl[eq_name] = lhs_consts | rhs_consts

    print("📋  CONSTANT INVENTORY")
    print("-"*38)
    for n, cs in tbl.items():
        print(f"{n:25s}: {pretty(cs) or '—'}")
    return tbl


def constant_overlap(tbl):
    """
    Given a dict {equation_name -> set_of_constants}, show the overlap matrix.
    """
    print("\n🔗  CONSTANT OVERLAP")
    print("-"*24)
    names = list(tbl.keys())
    N = len(names)
    M = np.zeros((N, N), int)

    for i, j in combinations(range(N), 2):
        M[i, j] = M[j, i] = len(tbl[names[i]] & tbl[names[j]])

    print("    " + "  ".join(f"{s[:3]}" for s in names))
    for i, row in enumerate(M):
        print(f"{names[i][:3]} " + "  ".join(f"{x:3d}" for x in row))


# ============================================================
# 5.  Coefficient-ratio scan
# ============================================================
def scan_coefficient_ratios(equations, tol=1e-6):
    """
    Look for special numeric ratios (like π, 2π, e, etc.) among
    the coefficients in every equation.
    """
    TARGETS = {
        np.pi      : "π",
        2*np.pi    : "2π",
        np.pi/2    : "π/2",
        np.e       : "e",
    }
    print("\n🔍  COEFFICIENT RATIO CHECKS")
    print("-"*40)

    for n, eq in equations.items():
        cf = coeffs(num(eq.lhs)) + coeffs(num(eq.rhs))
        for a, b in combinations(cf, 2):
            r = a / b
            for t, lab in TARGETS.items():
                # check ratio or inverse
                if abs(r - t) < tol or abs(r - 1/t) < tol:
                    print(f"{n:25s}: {a:.4g}/{b:.4g} ≈ {lab}")


# ============================================================
# 6.  Global PSLQ (if available)
# ============================================================
def global_pslq(equations, PSLQ_func, maxcoeff=10, maxterms=4):
    """
    Run PSLQ on all numeric coefficients extracted from
    all equations combined. If PSLQ is not available or
    no relation is found, a message is printed.
    """
    print("\n🔑  GLOBAL PSLQ RELATIONS (coeff ≤10)")
    print("-"*45)
    if PSLQ_func is None:
        print("  PSLQ not present in this runtime.")
        print("  ▶  To add it, run:  !pip install sympy-numerics  and re-start.")
        return

    # Gather all coefficients from all eqns
    allc = []
    for eq in equations.values():
        allc += coeffs(num(eq.lhs)) + coeffs(num(eq.rhs))

    # Try PSLQ
    rel = PSLQ_func(allc, maxcoeff=maxcoeff, maxterms=maxterms)
    if rel is None:
        print("  No short integer relation found.")
    else:
        # Make a nice string, skipping zero-coeff terms
        terms = [f"{c:+d}·{v:.6g}" for c, v in zip(rel, allc) if c]
        print("  " + " ".join(terms) + " = 0")


# ============================================================
# 7. Optional: A function to check dimensionless equations
# ============================================================
# A simple dimension-check routine can get complicated quickly
# (you'd need a full library of units). Here is a *stub* that
# just flags if any equation is purely symbolic in dimensionless form.

def is_dimensionless(expr, dimensionless_symbols=None):
    """
    Very naive dimension-check.
    Treats only 'dimensionless_symbols' as dimensionless.
    If the expression is purely numeric after substituting dimensionless=1,
    we call it dimensionless. This is a rough approach, *not* robust for real units.

    Example usage:
       is_dimensionless(sp.log(m*c**2), dimensionless_symbols=[sp.log])
       --> returns False (because m*c**2 won't vanish into a single numeric).
    """
    if dimensionless_symbols is None:
        dimensionless_symbols = []

    # Map dimensionless symbols to 1
    subs_map = {sym: 1 for sym in dimensionless_symbols}
    # Also map all known dimensionless constants (like pi, e, alpha) to 1
    # (You can refine which ones are truly dimensionless!)
    # We'll assume α is dimensionless, pi, e, etc.:
    for sym in [sp.pi, sp.E, α]:
        subs_map[sym] = 1

    # Attempt symbolic simplification
    expr_sub = expr.xreplace(subs_map)
    # If the result is a pure number, we consider it dimensionless
    return expr_sub.is_Number


def check_equations_dimensionless(equations):
    """
    Illustrative stub that tries is_dimensionless() on each
    LHS - RHS of the equations.
    This only 'works' if the user has declared certain
    symbols as dimensionless or not.
    """
    print("\n🔎  DIMENSIONLESS CHECK (very naive!)")
    print("-"*45)

    # Suppose we treat the following symbols as dimensionless:
    dimensionless_syms = [sp.log]  # example symbolic function

    for name, eq in equations.items():
        # check eq.lhs - eq.rhs
        difference = eq.lhs - eq.rhs
        if is_dimensionless(difference, dimensionless_symbols=dimensionless_syms):
            print(f"{name:25s}: Could be dimensionless.")
        else:
            print(f"{name:25s}: Not dimensionless (likely).")


# ============================================================
# 8.  Main demonstration
# ============================================================
if __name__ == "__main__":
    # (A) Inventory & Overlap
    tbl = constant_inventory(EQUATIONS)
    constant_overlap(tbl)

    # (B) Coefficient Ratio Scans
    scan_coefficient_ratios(EQUATIONS, tol=1e-6)

    # (C) PSLQ
    global_pslq(EQUATIONS, PSLQ)

    # (D) (Optional) Dimensionless check
    check_equations_dimensionless(EQUATIONS)

    print("\nDone.")

📋  CONSTANT INVENTORY
--------------------------------------
Einstein mass–energy     : c
Newton II                : —
Coulomb law              : 1/4, ε0, π
de Broglie λ             : c, ℏ
Schrödinger (free)       : -1, 1/2, ℏ
Planck E=hν              : h
Boltzmann kT             : k_B
Stefan–Boltzmann         : 1/60, c, ℏ, k_B, π, σ_SB
Einstein A↔B             : 1/8, c, h, π
Fine-structure α         : 1/4, α, c, e (charge), ε0, ℏ, π
Vacuum impedance         : c, μ0
Rydberg constant         : 1/2, α, c, h

🔗  CONSTANT OVERLAP
------------------------
    Ein  New  Cou  de   Sch  Pla  Bol  Ste  Ein  Fin  Vac  Ryd
Ein   0    0    0    1    0    0    0    1    1    1    1    1
New   0    0    0    0    0    0    0    0    0    0    0    0
Cou   0    0    0    0    0    0    0    1    1    3    0    0
de    1    0    0    0    1    0    0    2    1    2    1    1
Sch   0    0    0    1    0    0    0    1    0    1    0    1
Pla   0    0    0    0    0    0    0    0    1    0    0    1
Bo